# Phase 2 — YOLOv8 Floor Plan Segmentation Training
Run this notebook on **Google Colab** (T4 GPU) for full-dataset training.
Change runtime: Runtime → Change runtime type → T4 GPU

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────
!pip install ultralytics -q
!pip install zenodo-get -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 2. Download CubiCasa5k dataset ────────────────────────────────────────
# Dataset: https://zenodo.org/record/2613548
!zenodo_get 2613548 -o data/cubicasa5k
!ls data/cubicasa5k

In [ ]:
# ── 3. Clone your project repo ───────────────────────────────────────────
# Replace with your actual GitHub repo URL after pushing the code
!git clone https://github.com/YOUR_USERNAME/floor2model.git
%cd floor2model

In [ ]:
# ── 4. Prepare dataset in YOLO format ────────────────────────────────────
from src.segmentation.dataset import CubiCasaDataset

ds = CubiCasaDataset('data/cubicasa5k')
yaml_path = ds.prepare(output_dir='data/yolo_dataset')
print(f'Dataset ready: {yaml_path}')

In [ ]:
# ── 5. Train the model ────────────────────────────────────────────────────
from src.segmentation.trainer import SegmentationTrainer

trainer = SegmentationTrainer(
    dataset_yaml='data/yolo_dataset/dataset.yaml',
    model_size='s',        # 's' is best balance for this dataset
    epochs=100,
    batch_size=16,         # T4 GPU can handle batch 16 comfortably
    img_size=640,
    device='0',            # CUDA GPU in Colab
)

best_weights = trainer.train()
print(f'Best weights saved at: {best_weights}')

In [ ]:
# ── 6. Evaluate on test set ───────────────────────────────────────────────
metrics = trainer.validate(best_weights)
print(f"mAP50:    {metrics['map50']:.4f}")
print(f"mAP50-95: {metrics['map50_95']:.4f}")

# Expected results after 100 epochs:
# mAP50: ~0.72-0.78 for walls/doors/windows
# mAP50: ~0.55-0.65 for room types

In [ ]:
# ── 7. Export model for production ───────────────────────────────────────
# Export to ONNX for cross-platform deployment
onnx_path = trainer.export(best_weights, format='onnx')
print(f'ONNX model: {onnx_path}')

# Download best.pt to your local machine
from google.colab import files
files.download(best_weights)

In [ ]:
# ── 8. Quick inference test ───────────────────────────────────────────────
from src.segmentation.predictor import FloorPlanPredictor
from src.segmentation.visualizer import SegmentationVisualizer
import cv2
from google.colab.patches import cv2_imshow

predictor = FloorPlanPredictor(best_weights)
viz = SegmentationVisualizer()

# Test on a sample from the test set
test_image = 'data/yolo_dataset/images/test/'
import os
test_img = os.listdir(test_image)[0]

result = predictor.predict(test_image + test_img)
img = cv2.imread(test_image + test_img)
annotated = viz.draw(img, result)
cv2_imshow(annotated)